In [1]:
import pandas as pd

In [2]:
farm_df = pd.read_csv(r"C:\Users\Ксения\Downloads\farm.csv")
cat_df = pd.read_csv(r"C:\Users\Ксения\Downloads\Cat_only_rt.csv")

In [3]:
cat_df = cat_df[cat_df["year"] >= 1990]
farm_df = farm_df[farm_df["year"] <= 2022]

In [4]:
farm_df

,year,state,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased
0,1990,New South Wales,4073.0,206.0,206.0,18393.0,17123,976.0
1,1990,Northern Territory,0.0,0.0,0.0,15.0,4,15.0
2,1990,Queensland,3856.0,157.0,157.0,17240.0,17443,1023.0
3,1990,South Australia,2803.0,145.0,145.0,11211.0,10952,941.0
4,1990,Tasmania,641.0,34.0,34.0,2813.0,2589,86.0
...,...,...,...,...,...,...,...,...
226,2022,Queensland,1056.0,59.0,59.0,3267.0,3706,153.0
227,2022,South Australia,3003.0,123.0,123.0,9578.0,8842,452.0
228,2022,Tasmania,567.0,20.0,20.0,1804.0,1795,55.0
229,2022,Victoria,1608.0,50.0,50.0,4236.0,4239,548.0


In [5]:
cat_df

,year,stateTerritory,ibraRegion,forest2018Status,forest2013Status,capadStatus,speciesName,occurrenceCount
960,1990,Australian Capital Territory,Australian Alps,forest,forest,PA,Felis catus,1
961,1990,Australian Capital Territory,South Eastern Highlands,non-forest,non-forest,PA,Felis catus,1
962,1990,New South Wales,NSW North Coast,forest,forest,PA,Felis catus,1
963,1990,New South Wales,NSW North Coast,forest,forest,not protected,Felis catus,5
964,1990,New South Wales,South East Corner,non-forest,forest,not protected,Felis catus,1
...,...,...,...,...,...,...,...,...
4944,2022,Western Australia,Swan Coastal Plain,non-forest,non-forest,PA,Felis catus,1
4945,2022,Western Australia,Swan Coastal Plain,non-forest,non-forest,not protected,Felis catus,5
4946,2022,Western Australia,Warren,forest,forest,PA,Felis catus,1
4947,2022,Western Australia,Warren,non-forest,non-forest,PA,Felis catus,1


In [6]:
cat_df = cat_df.rename(columns={"stateTerritory": "state"})

In [7]:
cat_df["state"].unique()

array(['Australian Capital Territory', 'New South Wales',
       'Northern Territory', 'Queensland', 'South Australia', 'Victoria',
       'Unknown1', 'Western Australia', 'Tasmania'], dtype=object)

In [8]:
farm_df["state"].unique()

array(['New South Wales', 'Northern Territory', 'Queensland',
       'South Australia', 'Tasmania', 'Victoria', 'Western Australia'],
      dtype=object)

In [9]:
keys = ["year", "state"]

In [10]:
# sum of occurrenceCount for each (year, state)
cats_total = (
    cat_df.groupby(keys, as_index=False)["occurrenceCount"]
          .sum()
          .rename(columns={"occurrenceCount": "cats_occurrence_total"})
)

# forest/non-forest 2013 (sum of occurrenceCount)
cats_2013 = (
    cat_df.pivot_table(index=keys,
                       columns="forest2013Status",
                       values="occurrenceCount",
                       aggfunc="sum",
                       fill_value=0)
          .rename(columns=lambda c: f"cats_2013_{c.replace('-', '_')}")
          .reset_index()
)

# forest/non-forest 2018 (sum of occurrenceCount)
cats_2018 = (
    cat_df.pivot_table(index=keys,
                       columns="forest2018Status",
                       values="occurrenceCount",
                       aggfunc="sum",
                       fill_value=0)
          .rename(columns=lambda c: f"cats_2018_{c.replace('-', '_')}")
          .reset_index()
)

In [11]:
cats_state_year = (
    cats_total.merge(cats_2013, on=keys, how="outer")
              .merge(cats_2018, on=keys, how="outer")
)

In [12]:
cats_state_year

,year,state,cats_occurrence_total,cats_2013_forest,cats_2013_non_forest,cats_2018_forest,cats_2018_non_forest
0,1990,Australian Capital Territory,3,1,2,1,2
1,1990,New South Wales,69,48,21,52,17
2,1990,Northern Territory,43,11,32,14,29
3,1990,Queensland,17,9,8,7,10
4,1990,South Australia,26,9,17,12,14
...,...,...,...,...,...,...,...
203,2022,Queensland,78,19,59,16,62
204,2022,South Australia,150,20,130,14,136
205,2022,Tasmania,118,39,79,45,73
206,2022,Victoria,183,115,68,116,67


In [13]:
farm_cat_merged = farm_df.merge(cats_state_year, on=keys, how="left")
farm_cat_merged

,year,state,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased,cats_occurrence_total,cats_2013_forest,cats_2013_non_forest,cats_2018_forest,cats_2018_non_forest
0,1990,New South Wales,4073.0,206.0,206.0,18393.0,17123,976.0,69.0,48.0,21.0,52.0,17.0
1,1990,Northern Territory,0.0,0.0,0.0,15.0,4,15.0,43.0,11.0,32.0,14.0,29.0
2,1990,Queensland,3856.0,157.0,157.0,17240.0,17443,1023.0,17.0,9.0,8.0,7.0,10.0
3,1990,South Australia,2803.0,145.0,145.0,11211.0,10952,941.0,26.0,9.0,17.0,12.0,14.0
4,1990,Tasmania,641.0,34.0,34.0,2813.0,2589,86.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
226,2022,Queensland,1056.0,59.0,59.0,3267.0,3706,153.0,78.0,19.0,59.0,16.0,62.0
227,2022,South Australia,3003.0,123.0,123.0,9578.0,8842,452.0,150.0,20.0,130.0,14.0,136.0
228,2022,Tasmania,567.0,20.0,20.0,1804.0,1795,55.0,118.0,39.0,79.0,45.0,73.0
229,2022,Victoria,1608.0,50.0,50.0,4236.0,4239,548.0,183.0,115.0,68.0,116.0,67.0


In [14]:
farm_cat_merged.to_csv("farm_cat.csv", index=False)

In [15]:
# farm_cat_merged_from_2002 = farm_cat_merged_from_2002[farm_cat_merged_from_2002["year"] >= 2002]
# farm_cat_merged_from_2002.to_csv("farm_cat_from_2002.csv", index=False)